# TP 3 — Transfer learning avec MobileNetV2

**Objectifs**

- Charger un modèle pré-entraîné sur ImageNet (MobileNetV2).
- L'utiliser comme **extracteur de features** : geler tout le backbone et ne ré-entraîner que la dernière couche.
- Comparer à un entraînement from-scratch.

**Durée indicative : 50 minutes.**

> Note : on travaille sur 2 000 / 500 exemples de CIFAR-10 redimensionnés à 96×96 (MobileNetV2 demande au moins 96×96). Le premier appel téléchargera les poids pré-entraînés (~14 Mo).

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = "data"

# Stats ImageNet pour normaliser (le backbone a été entraîné dessus)
IM_MEAN, IM_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

## Exercice 1 — Données redimensionnées + normalisation ImageNet

Construis un transform :

```python
tf = transforms.Compose([
    transforms.Resize(96),
    transforms.ToTensor(),
    transforms.Normalize(IM_MEAN, IM_STD),
])
```

Charge CIFAR-10 (train et test) avec ce transform, prends 2 000 / 500 premiers exemples, crée les DataLoaders (batch 64 train, 128 test).

In [ ]:
# TODO


## Exercice 2 — Charger MobileNetV2 pré-entraîné

1. `model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)`.
2. **Gèle** tous les paramètres : `for p in model.parameters(): p.requires_grad = False`.
3. Remplace la dernière couche `model.classifier[1]` par une `nn.Linear(model.last_channel, 10)`. Cette couche **aura `requires_grad=True` par défaut**.
4. Affiche le nombre de paramètres entraînables vs total.

<details>
<summary>💡 Coup de pouce — charger MobileNetV2 pré-entraîné</summary>

**🎯 But :** charger un CNN pré-entraîné sur ImageNet et adapter sa tête pour CIFAR-10.

**Charger le modèle**

```python
from torchvision import models
model = models.mobilenet_v2(weights='IMAGENET1K_V1')
```

`weights='IMAGENET1K_V1'` télécharge les poids appris sur ImageNet (1.3M images, 1000 classes). Le modèle « sait déjà » détecter des bords, textures, formes, objets — utile pour CIFAR-10 même si les classes diffèrent.

**Inspecter l'architecture**

```python
print(model.classifier)
# Sequential(
#   (0): Dropout(p=0.2)
#   (1): Linear(in_features=1280, out_features=1000)
# )
print(model.last_channel)   # 1280 — sortie du backbone
```

MobileNetV2 a un **backbone** (extracteur de features) + un **classifier** (Sequential de Dropout + Linear vers 1000 classes ImageNet).

**Remplacer la tête pour 10 classes**

```python
import torch.nn as nn
model.classifier[1] = nn.Linear(model.last_channel, 10)
```

⚠️ **`model.classifier[1]`** : on remplace uniquement le Linear final, en gardant le Dropout. `model.last_channel` (= 1280) est l'attribut spécifique MobileNetV2 qui donne la dim de sortie du backbone.

**Geler le backbone (freeze)**

```python
for name, param in model.named_parameters():
    if 'classifier' not in name:
        param.requires_grad = False
```

Tous les params **sauf la nouvelle tête** sont gelés. Pendant le backward, leurs gradients ne seront pas calculés → pas d'update.

**Compter les paramètres trainables vs total**

```python
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainables : {trainable:,} / {total:,}  ({100*trainable/total:.1f} %)")
```

Avec backbone gelé : **~12 800 paramètres trainables** (juste la nouvelle tête Linear(1280→10)) contre **~3.5M au total**. On entraîne 0.4 % des params → ultra-rapide et data-efficient.

</details>

In [ ]:
# TODO


## Exercice 3 — Entraînement de la tête uniquement

Comme on n'optimise que la dernière couche, on passe à l'optimizer uniquement les paramètres avec `requires_grad=True` :

```python
trainable = filter(lambda p: p.requires_grad, model.parameters())
opt = torch.optim.Adam(trainable, lr=1e-3)
```

Entraîne 2 époques. Affiche l'accuracy sur le test.

<details>
<summary>💡 Coup de pouce — entraîner la tête uniquement</summary>

**🎯 But :** entraîner uniquement la nouvelle tête de classification, en gardant le backbone gelé.

**Adapter CIFAR-10 à la résolution MobileNetV2**

MobileNetV2 a été entraîné sur des images **224×224**. Pour CIFAR-10 (32×32), on doit upscale :

```python
transform = transforms.Compose([
    transforms.Resize(96),                    # upscale 32→96 (ImageNet utiliserait 224)
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],           # stats ImageNet (PAS CIFAR)
        std=[0.229, 0.224, 0.225],
    ),
])
```

⚠️ **Stats ImageNet, pas CIFAR** : le backbone a été normalisé avec ces stats lors de son pré-entraînement → on doit les **reproduire** pour qu'il fonctionne en mode inférence.

**Optimiser uniquement les params trainables**

```python
import torch.optim as optim
trainable_params = filter(lambda p: p.requires_grad, model.parameters())
optimizer = optim.Adam(trainable_params, lr=1e-3)
```

⚠️ **Pourquoi filtrer** : `optim.Adam(model.parameters())` essaierait d'updater **tous** les params, y compris les gelés. Sans `requires_grad`, ces gradients sont None → erreur ou silent fail. Le filter ne donne que les params trainables.

**Boucle d'entraînement (standard)**

```python
model.train()
for epoch in range(2):
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
```

**Résultats attendus**

Avec MobileNetV2 + 2 époques + CIFAR-10 :
- **Accuracy ≈ 65-75 %** typiquement
- Avec un from-scratch (sans transfer), 2 époques donneraient à peine 40-50 %
- Le transfer learning **double presque la performance** avec **bien moins d'entraînement**

**Pourquoi c'est si efficace**

Le backbone MobileNetV2 a déjà appris des **features visuelles génériques** sur ImageNet (textures, contours, parties d'objets). Ces features sont **réutilisables** pour CIFAR-10 même si les classes diffèrent — un CNN classifieur de chat sait déjà reconnaître des oreilles, des yeux, du poil.

**Pour aller plus loin : fine-tuning complet**

Après quelques époques de tête seule, on peut **dégeler progressivement** le backbone avec un `lr` plus petit (`1e-4` ou `1e-5`) → fine-tuning. Souvent +2-5 pts de plus.

</details>

In [ ]:
# TODO


## Exercice 4 — Comparaison à un from-scratch sur même budget

Reproduis l'expérience avec un MobileNetV2 **non pré-entraîné** (`weights=None`) entraîné from-scratch sur le même jeu et 2 époques. Compare les deux accuracies. Pourquoi le transfer learning gagne-t-il largement ?

In [ ]:
# TODO
